# Multiprocess a folder and summarize the flux timeseries (MCMC)

Point `data_folder` at a single day's folder of JSON chamber files. The cell below loads the raw data, runs the `Multiprocessor` in MCMC mode across CPU cores, selects the Pareto-optimal (deadband, cutoff) per measurement, and plots the resulting `dcdt(HM)` timeseries with a 16–84 percentile band.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('..')

In [ ]:
import logging
import pathlib

import matplotlib.pyplot as plt
import pandas as pd
import xarray as xr

from soilgasflux_fcs import Multiprocessor, json_reader

logging.getLogger('soilgasflux_fcs').setLevel(logging.INFO)

In [ ]:
# data_folder = pathlib.Path('/Users/alexnaokiasatokobayashi/Documents/data/Gals/2026_tpclass/low-cost sensor/2-2/data/2026-03-19')
data_folder = pathlib.Path('/Users/alexnaokiasatokobayashi/Documents/data/Kerzers/CHYN-Kerzers01/low-cost_sensor/kerzers_20260414/1-1/data/2026-04-02')
output_folder = pathlib.Path('/Users/alexnaokiasatokobayashi/Downloads/test_output')
output_folder.mkdir(parents=True, exist_ok=True)
chamber_id = data_folder.parent.parent.name

In [ ]:
initializer = json_reader.Initializer(folderPath=data_folder)
df = initializer.prepare_rawdata()
print(f'{df["id"].nunique()} measurements, {len(df)} rows')
df.head()

`run_MC` runs emcee per (deadband, cutoff) window, then `select_bestPareto` picks the Pareto-optimal window per timestamp and writes `<chamber_id>_<date>_bestPareto.nc` into `output_folder`.

In [ ]:
processor = Multiprocessor()
ds_mc = processor.run_MC(df=df, 
                         chamber_id=chamber_id, 
                         output_folder=str(output_folder), 
                         sensor_precision=30,
                         n_MC=8000)
ds_mc

In [ ]:
pareto_files = sorted(output_folder.glob(f'{chamber_id}_*_bestPareto.nc'))
ds_best = xr.open_mfdataset(pareto_files, combine='by_coords').sortby('time')
ds_best

In [ ]:
dcdt = ds_best['dcdt(HM)']
median = dcdt.median(dim='MC')
q16 = dcdt.quantile(0.16, dim='MC')
q84 = dcdt.quantile(0.84, dim='MC')

summary = pd.DataFrame({
    'time': ds_best['time'].values,
    'dcdt_median': median.values,
    'dcdt_q16': q16.values,
    'dcdt_q84': q84.values,
    'best_deadband': ds_best['best_deadband'].values,
    'best_cutoff': ds_best['best_cutoff'].values,
}).set_index('time')
summary.describe()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4), dpi=120)
ax.fill_between(summary.index, summary['dcdt_q16'], summary['dcdt_q84'],
                color='steelblue', alpha=0.3, label='16–84%')
ax.plot(summary.index, summary['dcdt_median'], color='steelblue',
        marker='o', ms=3, lw=0.8, label='median')
ax.set_xlabel('Time')
ax.set_ylabel('dC/dt (HM) [ppm / s]')
ax.set_title(f'{chamber_id} — {data_folder.name} (MCMC, Pareto-selected)')
ax.grid(alpha=0.3)
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()